# Load Library and Setup Envorpnment

In [ ]:
!pip install --upgrade transformers
!pip install datasets decord av accelerate openai-whisper moviepy

In [ ]:
from google.colab import drive
import torch
from transformers import AutoProcessor
from transformers.models.llava_next_video import LlavaNextVideoForConditionalGeneration

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Dataset

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("lmms-lab/AISG_Challenge")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Preprocess Dataset

In [ ]:
import pandas as pd
from IPython.display import display

# Prettify display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', 10)

# Convert and display
df = ds['test'].to_pandas()
#df = df.head(10)  # Limit to 10 samples for testing
display(df.head(10))

,qid,video_id,question_type,capability,question,duration,question_prompt,answer,youtube_url
0,0008-0,sj81PWrerDk,Primary Open-ended Question,Plot Attribute (Montage),What is the difference between the action of the last person in the video and the actions of the first two people?,8.85,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/sj81PWrerDk
1,0008-1,sj81PWrerDk,Paraphrased Open-ended Question,Plot Attribute (Montage),Can you describe how the actions of the last person in the video differ from other individuals?,8.85,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/sj81PWrerDk
2,0008-2,sj81PWrerDk,Correctly-led Open-ended Question,Plot Attribute (Montage),Did the last person open the bottle without using a knife?,8.85,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/sj81PWrerDk
3,0008-3,sj81PWrerDk,Wrongly-led Open-ended Question,Plot Attribute (Montage),Did the last person in the video open the bottle with a knife while the first two people failed in their attempts?,8.85,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/sj81PWrerDk
4,0008-7,sj81PWrerDk,Multiple-choice Question with a Single Correct Answer,Plot Attribute (Montage),How does the last person in the video open the bottle differently from the first two people?\nA. He uses a different type of knife.\nB. He uses a corkscrew.\nC. He uses a bottle opener.\nD. He uses his finger.,8.85,"E. None of the above\nSelect one best answer to the above multiple-choice question based on the video. Respond with only the letter (A, B, C, D or E) of the correct option.",,https://www.youtube.com/shorts/sj81PWrerDk
5,0012-0,AGCyLqLuUJ0,Primary Open-ended Question,Element Counting,How many robot figures appear in the video?,20.67,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/AGCyLqLuUJ0
6,0012-1,AGCyLqLuUJ0,Paraphrased Open-ended Question,Element Counting,What is the total number of robot figures shown in the video?,20.67,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/AGCyLqLuUJ0
7,0012-2,AGCyLqLuUJ0,Correctly-led Open-ended Question,Element Counting,Does the video display four robot figures in total?,20.67,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/AGCyLqLuUJ0
8,0012-3,AGCyLqLuUJ0,Wrongly-led Open-ended Question,Element Counting,Are there only two robot figures shown in the video?,20.67,Please state your answer with a brief explanation.,,https://www.youtube.com/shorts/AGCyLqLuUJ0
9,0012-7,AGCyLqLuUJ0,Multiple-choice Question with a Single Correct Answer,Element Counting,How many robot figures are shown in the video?\nA. Two\nB. Three\nC. Four\nD. Five,20.67,"E. None of the above\nSelect one best answer to the above multiple-choice question based on the video. Respond with only the letter (A, B, C, D or E) of the correct option.",,https://www.youtube.com/shorts/AGCyLqLuUJ0


# Load Model

**LLaVA**

In [ ]:
processor = AutoProcessor.from_pretrained("llava-hf/LLaVA-NeXT-Video-7B-hf")
model = LlavaNextVideoForConditionalGeneration.from_pretrained("llava-hf/LLaVA-NeXT-Video-7B-hf", device_map="auto", torch_dtype=torch.float16)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

# Evalaution

In [ ]:
import os
import numpy as np
import av
import time
from tqdm import tqdm
from PIL import Image
from datetime import timedelta
from moviepy.editor import VideoFileClip
import whisper

# Load Whisper on CPU to avoid CUDA OOM
whisper_model = whisper.load_model("medium", device="cuda")

# Extract multiple frames evenly from video
def extract_frames(video_path, num_frames=8):
    container = av.open(video_path)
    stream = container.streams.video[0]
    total_frames = stream.frames
    indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i in indices:
            frames.append(frame.to_image().convert("RGB"))
        if len(frames) == num_frames:
            break
    return frames

# Extract audio and transcribe with Whisper
def clean_transcript(text):
    import re
    return re.sub(r"<.*?>", "", text).replace("  ", " ").strip()

def transcribe_audio(video_path):
    global whisper_model
    audio_path = video_path.replace(".mp4", ".wav")
    try:
        clip = VideoFileClip(video_path)
        clip.audio.write_audiofile(audio_path, verbose=False, logger=None)
        try:
            result = whisper_model.transcribe(audio_path)
        except torch.cuda.OutOfMemoryError:
            print("⚠️ Whisper OOM — switching to CPU for this sample.")
            torch.cuda.empty_cache()
            whisper_model = whisper.load_model("medium", device="cpu")
            result = whisper_model.transcribe(audio_path)
            whisper_model = whisper.load_model("medium", device="cuda")
        text = clean_transcript(result["text"].strip())
        if len(text.split()) < 5:
            return None
        return text
    except Exception as e:
        print(f"⚠️ Audio error: {e}")
        return None

# Path to video directory
video_base_path = "/content/drive/MyDrive/Benchmark-AllVideos-HQ-Encoded-challenge"

# Prepare results container
results = []
processing_times = []
total_start = time.time()

# Find actual matching column names
video_col = next((c for c in df.columns if 'video' in c.lower()), None)
question_col = next((c for c in df.columns if 'question' in c.lower()), None)
question_id_col = next((c for c in df.columns if 'id' in c.lower()), None)

print(f"Using columns: video_id={video_col}, question={question_col}, question_id={question_id_col}")

# Loop through each test sample
for idx, row in enumerate(df.itertuples(index=False), start=1):
    video_id = getattr(row, video_col, None)
    actual_question = getattr(row, "question", "")
    question_type = getattr(row, "question_type", "")
    question_prompt = getattr(row, "question_prompt", "")
    capability = getattr(row, "capability", "")

    question = f"""{actual_question}
Instruction: {question_prompt}
Type: {question_type}
Capability: {capability}"""
    qid = getattr(row, question_id_col, None)
    if video_id is None or question is None:
        print(f"\u26a0\ufe0f Skipping video {idx} due to missing question or video_id")
        continue
    video_path = os.path.join(video_base_path, video_id + ".mp4")

    clip_start = time.time()
    try:
        # Extract multiple frames
        frames = extract_frames(video_path)
        if not frames:
            raise ValueError("No frames extracted")

        # Transcribe audio if useful
        transcript = transcribe_audio(video_path)
        if transcript:
            from transformers import pipeline
            summarizer = pipeline("summarization", device="cpu")
            summary = summarizer(transcript, max_length=80, min_length=20, do_sample=False)[0]['summary_text']
            question += f"""
Note: This is a summary of the video's transcript. Use it only if it helps.
Transcript Summary: {summary}
Please answer clearly and do not repeat the question.
"""

        # Format input as chat-style prompt
        conversation = [{
            "role": "user",
            "content": [
                {"type": "text", "text": f'''{question}

This is a visual question. Please refer to what's happening in the video frames.'''},
                {"type": "video"}
            ],
        }]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

        # Prepare input for LLaVA-NeXT
        inputs = processor(text=prompt, videos=frames, return_tensors="pt", padding=True).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=150)
        response = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()

        # Clean up assistant prefix
        if "ASSISTANT:" in response:
            response = response.split("ASSISTANT:", 1)[-1].strip()

        # Soft check for vague or unhelpful responses
        if "correct answer" in question.lower() and response.strip().endswith("?"):
            response = "I am unable to determine the correct answer from the options provided."

        # Add grounding logic: encourage the model to provide direct answers
        if response.strip().lower().startswith("what") or response.strip().endswith("?"):
            response += " Please answer based on what is shown or said in the video."

    except Exception as e:
        print(f"\u274c Error processing video {idx} ({video_id}): {e}")
        response = "[ERROR]"

    # Save result
    results.append({"question_id": qid, "pred": response})

    # Estimate time remaining
    elapsed = time.time() - clip_start
    processing_times.append(elapsed)
    avg_time = np.mean(processing_times)
    remaining = avg_time * (len(df) - idx)

    print(f"\n[Video {idx}/{len(df)}]")
    print(f"[Time taken: {elapsed:.2f}s]")
    print(f"[ETA: {str(timedelta(seconds=int(remaining)))}]")
    print(f"[Q] {question}")
    print(f"[A] {response}")
    print("-" * 50)

    # Free up memory
    for var in ["frames", "inputs", "outputs"]:
        if var in locals():
            del locals()[var]
    torch.cuda.empty_cache()

# Save to CSV
result_df = pd.DataFrame(results)
result_df.to_csv("/content/drive/MyDrive/tiktok_submission.csv", index=False)

# Completion time summary
total_time = time.time() - total_start
print(f"\n\u2705 All done in {total_time/60:.2f} minutes ({str(timedelta(seconds=int(total_time)))})")

Using columns: video_id=video_id, question=question_type, question_id=qid

[Video 1/1500]
[Time taken: 6.72s]
[ETA: 2:47:47]
[Q] What is the difference between the action of the last person in the video and the actions of the first two people?
Instruction: Please state your answer with a brief explanation.
Type: Primary Open-ended Question
Capability: Plot Attribute (Montage)
[A] The last person in the video is holding a bottle of Coca-Cola and appears to be about to drink it, while the first two people are holding a bottle of Coca-Cola and a can of Coca-Cola, respectively, and they are not drinking from them. The difference is that the last person is about to drink, while the first two people are not drinking at the moment.
--------------------------------------------------

[Video 2/1500]
[Time taken: 5.89s]
[ETA: 2:37:22]
[Q] Can you describe how the actions of the last person in the video differ from other individuals?
Instruction: Please state your answer with a brief explanation.

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu



[Video 6/1500]
[Time taken: 19.43s]
[ETA: 2:48:19]
[Q] How many robot figures appear in the video?
Instruction: Please state your answer with a brief explanation.
Type: Primary Open-ended Question
Capability: Element Counting
Note: This is a summary of the video's transcript. Use it only if it helps.
Transcript Summary:  Students   -  ‘When you see the girl!’: ‘I’m like when you see a girl.’. ‘‘’�, ‘We’ll be happy to see you!‘'’,’ says one student .
Please answer clearly and do not repeat the question.

[A] There are two robot figures in the video.
--------------------------------------------------


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu



[Video 7/1500]
[Time taken: 20.96s]
[ETA: 3:38:40]
[Q] What is the total number of robot figures shown in the video?
Instruction: Please state your answer with a brief explanation.
Type: Paraphrased Open-ended Question
Capability: Element Counting
Note: This is a summary of the video's transcript. Use it only if it helps.
Transcript Summary:      грраг -  - ‘I’m obliged to’ve been under a dictatorship.’�s rule of law .’s ‘I'm obliged to be under’,’ he said. ‘‘’ ’.‘ ‚
Please answer clearly and do not repeat the question.

[A] There are a total of 4 robot figures shown in the video.
--------------------------------------------------

[Video 8/1500]
[Time taken: 20.45s]
[ETA: 4:14:47]
[Q] Does the video display four robot figures in total?
Instruction: Please state your answer with a brief explanation.
Type: Correctly-led Open-ended Question
Capability: Element Counting
[A] Yes, the video displays four robot figures in total.
--------------------------------------------------

[Video 9/1

...


[Video 1493/1500]
[Time taken: 7.12s]
[ETA: 0:01:58]
[Q] Does the woman's mood at the end of the video reflect frustration or anger due to the man's pretending to be a king during her yoga session?
Instruction: Please state your answer with a brief explanation.
Type: Correctly-led Open-ended Question
Capability: Character Reaction Causality
Note: This is a summary of the video's transcript. Use it only if it helps.
Transcript Summary:  Oh, your eyes Your eyes Your Eyes, Your eyes, Your Eyes . Your eyes are the eyes of the world . Your Eyes are the world's eyes, the eyes are of the eyes .
Please answer clearly and do not repeat the question.

[A] No
--------------------------------------------------


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Your max_length is set to 80, but your input_length is only 14. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)



[Video 1494/1500]
[Time taken: 8.84s]
[ETA: 0:01:41]
[Q] Is the woman's mood at the end of the video one of confusion and curiosity, as she failed to undertand the man's action completely?
Instruction: Please state your answer with a brief explanation.
Type: Wrongly-led Open-ended Question
Capability: Character Reaction Causality
Note: This is a summary of the video's transcript. Use it only if it helps.
Transcript Summary:  Oh, your eyes Your eyes Your Eyes, Your eyes, Your Eyes . Your eyes are the eyes of the world . Your Eyes are the world's eyes, the eyes are of the eyes .
Please answer clearly and do not repeat the question.

[A] Yes, the woman's mood at the end of the video appears to be one of confusion and curiosity, as she seems to be unsure about the man's actions and is looking at him with a questioning expression.
--------------------------------------------------


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Your max_length is set to 80, but your input_length is only 14. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)



[Video 1495/1500]
[Time taken: 7.20s]
[ETA: 0:01:24]
[Q] What were the reasons for the woman's mood at the end of the video?
A. She was curious about what happened.
B. The man was not taking care of their child.
C. She failed to understand the man's actions.
D. The man made it look like she was doing a kowtow while the man preteded to be a king.
Instruction: E. None of the above
Select one best answer to the above multiple-choice question based on the video. Respond with only the letter (A, B, C, D or E) of the correct option.
Type: Multiple-choice Question with a Single Correct Answer
Capability: Character Reaction Causality
Note: This is a summary of the video's transcript. Use it only if it helps.
Transcript Summary:  Oh, your eyes Your eyes Your Eyes, Your eyes, Your Eyes . Your eyes are the eyes of the world . Your Eyes are the world's eyes, the eyes are of the eyes .
Please answer clearly and do not repeat the question.

[A] D
--------------------------------------------------

